In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import math


# ══════════════════════════════════════════════════════════════════════════════
# 1.  OPTICAL SIMULATOR  — σ=1.5 (moderate defocus)
# ══════════════════════════════════════════════════════════════════════════════

class OpticalSimulator(nn.Module):
    def __init__(self, sigma=1.5, vignette_strength=0.4):
        super().__init__()
        ax = torch.arange(-6., 7.)
        xx, yy = torch.meshgrid(ax, ax, indexing='ij')
        kernel  = torch.exp(-(xx**2 + yy**2) / (2. * sigma**2))
        kernel /= kernel.sum()
        self.register_buffer('kernel', kernel.view(1, 1, 13, 13))
        self.vignette_strength = vignette_strength

    def forward(self, x):
        blurred = torch.cat([
            F.conv2d(x[:, i:i+1], self.kernel, padding=6)
            for i in range(x.shape[1])
        ], dim=1)
        B, C, H, W = blurred.shape
        yg, xg = torch.meshgrid(
            torch.linspace(-1, 1, H, device=x.device),
            torch.linspace(-1, 1, W, device=x.device),
            indexing='ij')
        mask = (1 - self.vignette_strength * (xg**2 + yg**2).sqrt()).clamp(0, 1)
        return blurred * mask.view(1, 1, H, W).expand(B, C, H, W)


# ══════════════════════════════════════════════════════════════════════════════
# 2.  OPTICAL DATASET
# ══════════════════════════════════════════════════════════════════════════════

class OpticalDataset(Dataset):
    def __init__(self, base_dataset, num_samples=None, blur_prob=0.5,
                 dataset_name='cifar10', seed=42, sigma=1.5):
        self.base_dataset = base_dataset
        self.blur_prob    = blur_prob
        self.dataset_name = dataset_name
        self.rng          = np.random.RandomState(seed)
        self.optical_sim  = OpticalSimulator(sigma=sigma, vignette_strength=0.4)

        if dataset_name == 'cifar10':
            self.mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
            self.std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
        else:
            self.mean = torch.tensor([0.1307]).view(1, 1, 1)
            self.std  = torch.tensor([0.3081]).view(1, 1, 1)

        self.channels = 3 if dataset_name == 'cifar10' else 1
        n = len(base_dataset)
        if num_samples is None or num_samples >= n:
            self.indices = np.arange(n)
        else:
            self.indices = self.rng.choice(n, num_samples, replace=False)
        self.blur_flags = self.rng.rand(len(self.indices)) < blur_prob

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_pil, label = self.base_dataset[self.indices[idx]]
        img = transforms.ToTensor()(img_pil)
        if self.blur_flags[idx]:
            img = self.optical_sim(img.unsqueeze(0)).squeeze(0)
        if self.channels == 1 and img.shape[0] == 1:
            img = img.repeat(3, 1, 1)
        img = (img - self.mean) / self.std
        return img, label


# ══════════════════════════════════════════════════════════════════════════════
# 3.  BACKBONES
# ══════════════════════════════════════════════════════════════════════════════

class MNISTBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))

    def forward(self, x):
        return self.features(x).flatten(1)


class CIFAR10Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),    nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),   nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),  nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))

    def forward(self, x):
        return self.features(x).flatten(1)


# ══════════════════════════════════════════════════════════════════════════════
# 4.  BLUR ESTIMATOR + PROMPT GENERATOR
# ══════════════════════════════════════════════════════════════════════════════

class BlurEstimator(nn.Module):
    def __init__(self, blur_dim=32, cnn_dim=16):
        super().__init__()
        lap = torch.tensor([[0.,1.,0.],[1.,-4.,1.],[0.,1.,0.]]).view(1,1,3,3)
        sx  = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]]).view(1,1,3,3)
        sy  = torch.tensor([[-1.,-2.,-1.],[0.,0.,0.],[1.,2.,1.]]).view(1,1,3,3)
        self.register_buffer('laplacian', lap)
        self.register_buffer('sobel_x', sx)
        self.register_buffer('sobel_y', sy)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, cnn_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(cnn_dim), nn.ReLU(inplace=True),
            nn.Conv2d(cnn_dim, cnn_dim, 3, padding=1, stride=2, bias=False),
            nn.BatchNorm2d(cnn_dim), nn.ReLU(inplace=True),
            nn.Conv2d(cnn_dim, cnn_dim, 3, padding=1, stride=2, bias=False),
            nn.BatchNorm2d(cnn_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1))
        self.fusion = nn.Sequential(
            nn.Linear(2 + cnn_dim, blur_dim),
            nn.LayerNorm(blur_dim), nn.ReLU(inplace=True))

    def _sharpness(self, x):
        gray     = x.mean(dim=1, keepdim=True)
        lap_var  = F.conv2d(gray, self.laplacian, padding=1).var(dim=[2,3]).squeeze(1)
        gx       = F.conv2d(gray, self.sobel_x, padding=1)
        gy       = F.conv2d(gray, self.sobel_y, padding=1)
        grad_var = (gx**2 + gy**2 + 1e-8).sqrt().var(dim=[2,3]).squeeze(1)
        return torch.stack([torch.log1p(lap_var), torch.log1p(grad_var)], dim=1)

    def forward(self, x):
        return self.fusion(torch.cat([self._sharpness(x), self.cnn(x).flatten(1)], dim=1))


class PromptGenerator(nn.Module):
    def __init__(self, blur_dim=32, feature_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(blur_dim, 64),  nn.GELU(), nn.LayerNorm(64),
            nn.Linear(64, 128),       nn.GELU(), nn.LayerNorm(128),
            nn.Linear(128, feature_dim))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, d):
        return self.net(d)


# ══════════════════════════════════════════════════════════════════════════════
# 5.  CLASSIFIERS
# ══════════════════════════════════════════════════════════════════════════════

class BaselineClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes=10):
        super().__init__()
        self.backbone   = backbone
        self.dropout    = nn.Dropout(0.5)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.classifier(self.dropout(self.backbone(x)))


class ICOPClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes=10,
                 dataset_name='cifar10', blur_dim=32):
        super().__init__()
        self.backbone         = backbone
        self.feature_dim      = feature_dim
        self.dataset_name     = dataset_name
        self.blur_estimator   = BlurEstimator(blur_dim=blur_dim)
        self.prompt_generator = PromptGenerator(blur_dim=blur_dim, feature_dim=feature_dim)
        self.dropout          = nn.Dropout(0.5)
        self.classifier       = nn.Linear(feature_dim, num_classes)

    def evolve_prompt(self, clean_loader, device, iterations=400,
                      contrastive_margin=0.5, sigma=1.5):
        self.eval()
        self.blur_estimator.train()
        self.prompt_generator.train()
        for p in self.backbone.parameters():   p.requires_grad = False
        for p in self.classifier.parameters(): p.requires_grad = False

        opt_sim = OpticalSimulator(sigma=sigma, vignette_strength=0.4).to(device)

        mv, sv = ([0.4914,0.4822,0.4465], [0.2470,0.2435,0.2616]) \
                 if self.dataset_name == 'cifar10' else ([0.1307]*3, [0.3081]*3)
        mean = torch.tensor(mv).view(1,3,1,1).to(device)
        std  = torch.tensor(sv).view(1,3,1,1).to(device)

        opt = optim.Adam(list(self.blur_estimator.parameters()) +
                         list(self.prompt_generator.parameters()),
                         lr=5e-4, weight_decay=1e-4)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iterations)

        best_loss = float('inf')
        best_e = {k: v.clone() for k,v in self.blur_estimator.state_dict().items()}
        best_g = {k: v.clone() for k,v in self.prompt_generator.state_dict().items()}

        print(f"  Evolving ICOP (σ={sigma}, margin={contrastive_margin}, iters={iterations})...")

        it = 0
        while it < iterations:
            for images, _ in clean_loader:
                if it >= iterations: break
                images = images.to(device)
                unnorm = (images * std + mean).clamp(0,1)
                with torch.no_grad():
                    blurred      = (opt_sim(unnorm) - mean) / std
                    clean_feat   = self.backbone(images)
                    blurred_feat = self.backbone(blurred)

                desc_b    = self.blur_estimator(blurred)
                prompt    = self.prompt_generator(desc_b)
                corrected = blurred_feat + prompt

                l_corr = F.mse_loss(clean_feat, corrected)
                l_cos  = 1.0 - F.cosine_similarity(clean_feat, corrected, dim=1).mean()
                l_mag  = F.mse_loss(torch.norm(corrected,dim=1), torch.norm(clean_feat,dim=1))

                desc_c   = self.blur_estimator(images)
                p_clean  = self.prompt_generator(desc_c)
                l_sup    = (p_clean ** 2).mean()

                nb = torch.norm(prompt,  dim=1)
                nc = torch.norm(p_clean, dim=1)
                l_margin = F.relu(nc - nb + contrastive_margin).mean()

                loss = (10.0*l_corr + 5.0*l_cos + 2.0*l_mag +
                        5.0*l_sup + 10.0*l_margin)

                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.blur_estimator.parameters()) +
                    list(self.prompt_generator.parameters()), 1.0)
                opt.step()
                sch.step()

                if loss.item() < best_loss:
                    best_loss = loss.item()
                    best_e = {k:v.clone() for k,v in self.blur_estimator.state_dict().items()}
                    best_g = {k:v.clone() for k,v in self.prompt_generator.state_dict().items()}

                it += 1
                if it % 100 == 0:
                    with torch.no_grad():
                        raw  = F.mse_loss(clean_feat, blurred_feat).item()
                        red  = (1 - l_corr.item()/raw)*100 if raw > 0 else 0
                    print(f"    Iter {it:4d}/{iterations} | Loss:{loss.item():.4f} | "
                          f"corr={l_corr.item():.4f} sup={l_sup.item():.4f} "
                          f"margin={l_margin.item():.4f} | "
                          f"‖p_blur‖={nb.mean().item():.3f} "
                          f"‖p_cln‖={nc.mean().item():.3f} "
                          f"(gap={nb.mean().item()-nc.mean().item():+.3f}) | "
                          f"Err↓:{red:.1f}%")

        self.blur_estimator.load_state_dict(best_e)
        self.prompt_generator.load_state_dict(best_g)
        for p in self.backbone.parameters():   p.requires_grad = True
        for p in self.classifier.parameters(): p.requires_grad = True
        print(f"\n  ✓ ICOP evolved (best loss: {best_loss:.4f})\n")

    def forward(self, x):
        d = self.blur_estimator(x)
        p = self.prompt_generator(d)
        return self.classifier(self.dropout(self.backbone(x) + p))


# ══════════════════════════════════════════════════════════════════════════════
# 6.  TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer, criterion, device, desc=""):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.1%}'})
    return total_loss / len(loader), correct / total


def evaluate_with_preds(model, loader, device, desc=""):
    model.eval()
    results = []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)
            results.append((model(images).argmax(1) == labels).cpu().numpy())
    arr = np.concatenate(results)
    return arr.mean(), arr


def mcnemar_test(a, b):
    bc = int(np.sum( a & ~b))
    cb = int(np.sum(~a &  b))
    if bc + cb == 0: return 0.0, 1.0
    chi2  = (abs(bc - cb) - 1.0)**2 / (bc + cb)
    return chi2, 1.0 - math.erf(math.sqrt(chi2 / 2.0))


def finetune_frozen(model, mixed_ld, val_bld, criterion, device,
                    epochs=30, desc="FT", extra_params=None):
    """
    Fine-tune a model with backbone frozen.
    Returns best (val_blurred_acc, per_sample_correct_array).
    extra_params: additional parameter groups beyond the classifier head.
    """
    for p in model.backbone.parameters():
        p.requires_grad = False

    param_groups = [{'params': model.classifier.parameters(), 'lr': 5e-4}]
    if extra_params:
        param_groups += extra_params

    opt = optim.AdamW(param_groups, weight_decay=1e-3)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_acc   = 0.0
    best_state = None

    for e in range(epochs):
        train_epoch(model, mixed_ld, opt, criterion, device, f"{desc} {e+1}/{epochs}")
        sch.step()
        vb_acc, _ = evaluate_with_preds(model, val_bld, device, "ValBlur")
        if vb_acc > best_acc:
            best_acc   = vb_acc
            best_state = {
                k: v.clone() for k, v in model.state_dict().items()
                if not k.startswith('backbone.')   # save everything except frozen backbone
            }
        if (e + 1) % 10 == 0:
            print(f"  [{desc}] Epoch {e+1}: ValBlur={vb_acc:.1%} (best={best_acc:.1%})")

    # Restore best non-backbone state
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current)

    for p in model.backbone.parameters():
        p.requires_grad = True

    return evaluate_with_preds(model, val_bld, device, f"{desc} Final Blurred")


# ══════════════════════════════════════════════════════════════════════════════
# 7.  VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def validate_icop(dataset_name='cifar10', num_val=5000, device=None,
                  blur_dim=32, sigma=1.5):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'='*80}\n🚀 VALIDATING ICOP v7 ON {dataset_name.upper()} (σ={sigma})\n{'='*80}\n")

    if dataset_name == 'mnist':
        train_raw, test_raw = (datasets.MNIST('./data', train=t, download=True) for t in [True,False])
        feature_dim, backbone_cls = 128, MNISTBackbone
    elif dataset_name == 'fashion':
        train_raw, test_raw = (datasets.FashionMNIST('./data', train=t, download=True) for t in [True,False])
        feature_dim, backbone_cls = 128, MNISTBackbone
    else:
        train_raw, test_raw = (datasets.CIFAR10('./data', train=t, download=True) for t in [True,False])
        feature_dim, backbone_cls = 256, CIFAR10Backbone

    print(f"Train: {len(train_raw)} | Test: {len(test_raw)} | Val: {num_val} per condition | σ={sigma}")

    kw = dict(dataset_name=dataset_name, sigma=sigma)
    clean_ld  = DataLoader(OpticalDataset(train_raw, blur_prob=0.0, seed=0, **kw),
                           128, shuffle=True,  num_workers=2, pin_memory=True)
    evolve_ld = DataLoader(OpticalDataset(train_raw, blur_prob=0.0, seed=1, **kw),
                           128, shuffle=True,  num_workers=2, pin_memory=True)
    mixed_ld  = DataLoader(OpticalDataset(train_raw, blur_prob=0.5, seed=2, **kw),
                           128, shuffle=True,  num_workers=2, pin_memory=True)
    val_cld   = DataLoader(OpticalDataset(test_raw, num_samples=num_val, blur_prob=0.0, seed=1337, **kw),
                           128, shuffle=False, num_workers=2, pin_memory=True)
    val_bld   = DataLoader(OpticalDataset(test_raw, num_samples=num_val, blur_prob=1.0, seed=1337, **kw),
                           128, shuffle=False, num_workers=2, pin_memory=True)

    criterion = nn.CrossEntropyLoss()
    def lr_lambda(e):
        if e < 10: return (e+1)/10
        return 0.5**((e-10)//20)

    # ════════════════════════════════════════
    # PHASE 1 — BASELINE  (clean pretraining)
    # ════════════════════════════════════════
    print("\n[1/5] BASELINE: clean pretraining (60 epochs)...")
    baseline = BaselineClassifier(backbone_cls(), feature_dim).to(device)
    opt = optim.AdamW([
        {'params': baseline.backbone.parameters(),   'lr': 2e-4},
        {'params': baseline.classifier.parameters(), 'lr': 1e-3}
    ], weight_decay=5e-4)
    sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    for e in range(60):
        loss, acc = train_epoch(baseline, clean_ld, opt, criterion, device, f"Base {e+1}/60")
        sch.step()
        if (e+1) % 15 == 0: print(f"  Epoch {e+1}: Loss={loss:.4f}, Acc={acc:.1%}")

    base_cln_acc, base_cln_c = evaluate_with_preds(baseline, val_cld, device, "Base Clean")
    base_blr_acc, base_blr_c = evaluate_with_preds(baseline, val_bld, device, "Base Blurred")
    base_deg = (base_cln_acc - base_blr_acc) / base_cln_acc * 100
    print(f"\n  ✓ Baseline (clean-FT): Clean={base_cln_acc:.1%}, "
          f"Blurred={base_blr_acc:.1%}, Degradation={base_deg:.1f}%\n")

    # ════════════════════════════════════════
    # PHASE 2 — FT BASELINE (ablation)
    #   Same pretrained backbone, frozen,
    #   classifier head fine-tuned on mixed data.
    #   This is the null hypothesis for ICOP.
    # ════════════════════════════════════════
    print("[2/5] FT-BASELINE (ablation): fine-tune head on mixed data, NO prompt...")
    import copy
    ft_baseline = BaselineClassifier(copy.deepcopy(baseline.backbone), feature_dim).to(device)
    ft_baseline.classifier.load_state_dict(baseline.classifier.state_dict())

    ft_blr_acc, ft_blr_c = finetune_frozen(
        ft_baseline, mixed_ld, val_bld, criterion, device,
        epochs=30, desc="FT-Base")
    ft_cln_acc, ft_cln_c = evaluate_with_preds(ft_baseline, val_cld, device, "FT-Base Clean")
    ft_deg = (ft_cln_acc - ft_blr_acc) / ft_cln_acc * 100
    print(f"\n  ✓ FT-Baseline (mixed head): Clean={ft_cln_acc:.1%}, "
          f"Blurred={ft_blr_acc:.1%}, Degradation={ft_deg:.1f}%\n")

    # ════════════════════════════════════════
    # PHASE 3 — ICOP PRETRAINING (clean)
    # ════════════════════════════════════════
    print("[3/5] ICOP: clean pretraining (60 epochs)...")
    import copy

    icop = ICOPClassifier(
        copy.deepcopy(baseline.backbone),   # ← same weights as FT-Baseline
        feature_dim,
        dataset_name=dataset_name,
        blur_dim=blur_dim
    ).to(device)
    icop.classifier.load_state_dict(baseline.classifier.state_dict())
    opt = optim.AdamW([
        {'params': icop.backbone.parameters(),         'lr': 2e-4},
        {'params': icop.classifier.parameters(),       'lr': 1e-3},
        {'params': icop.blur_estimator.parameters(),   'lr': 5e-5},
        {'params': icop.prompt_generator.parameters(), 'lr': 5e-5},
    ], weight_decay=5e-4)
    sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    for e in range(60):
        loss, acc = train_epoch(icop, clean_ld, opt, criterion, device, f"ICOP pretrain {e+1}/60")
        sch.step()
        if (e+1) % 15 == 0: print(f"  Epoch {e+1}: Loss={loss:.4f}, Acc={acc:.1%}")
    print("  ✓ ICOP pretraining complete\n")

    # ════════════════════════════════════════
    # PHASE 4 — PROMPT EVOLUTION
    # ════════════════════════════════════════
    print("[4/5] Evolving input-conditioned prompt...")
    margin = {'mnist': 0.3, 'fashion': 0.4, 'cifar10': 0.6}.get(dataset_name, 0.5)
    icop.evolve_prompt(evolve_ld, device, iterations=400,
                       contrastive_margin=margin, sigma=sigma)

    # ════════════════════════════════════════
    # PHASE 5 — ICOP FINE-TUNING (frozen backbone, mixed data)
    # ════════════════════════════════════════
    print("[5/5] ICOP fine-tuning: backbone FROZEN, mixed data (30 epochs)...")
    icop_blr_acc, icop_blr_c = finetune_frozen(
        icop, mixed_ld, val_bld, criterion, device,
        epochs=30, desc="ICOP-FT",
        extra_params=[
            {'params': icop.blur_estimator.parameters(),   'lr': 1e-4},
            {'params': icop.prompt_generator.parameters(), 'lr': 1e-4},
        ])
    icop_cln_acc, icop_cln_c = evaluate_with_preds(icop, val_cld, device, "ICOP Clean")
    icop_deg = (icop_cln_acc - icop_blr_acc) / icop_cln_acc * 100
    print(f"\n  ✓ ICOP: Clean={icop_cln_acc:.1%}, Blurred={icop_blr_acc:.1%}, "
          f"Degradation={icop_deg:.1f}%\n")

    # ── Statistics ────────────────────────────────────────────────────
    # Primary comparison: ICOP vs FT-Baseline (isolates prompt contribution)
    chi2_icop_vs_ft,   p_icop_vs_ft   = mcnemar_test(ft_blr_c,   icop_blr_c)
    # Secondary: ICOP vs clean Baseline
    chi2_icop_vs_base, p_icop_vs_base = mcnemar_test(base_blr_c, icop_blr_c)
    # Sanity: FT-Baseline vs clean Baseline
    chi2_ft_vs_base,   p_ft_vs_base   = mcnemar_test(base_blr_c, ft_blr_c)

    return {
        'dataset':        dataset_name,
        'baseline':       {'clean': base_cln_acc, 'blurred': base_blr_acc, 'deg': base_deg},
        'ft_baseline':    {'clean': ft_cln_acc,   'blurred': ft_blr_acc,   'deg': ft_deg},
        'icop':           {'clean': icop_cln_acc,  'blurred': icop_blr_acc, 'deg': icop_deg},
        # Primary: does prompt add value beyond mixed fine-tuning?
        'icop_vs_ft':     {'delta': icop_blr_acc - ft_blr_acc,   'p': p_icop_vs_ft},
        # Secondary: overall improvement from clean baseline
        'icop_vs_base':   {'delta': icop_blr_acc - base_blr_acc, 'p': p_icop_vs_base},
        'ft_vs_base':     {'delta': ft_blr_acc - base_blr_acc,   'p': p_ft_vs_base},
    }


# ══════════════════════════════════════════════════════════════════════════════
# 8.  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SIGMA  = 1.5   # Moderate defocus — realistic optical degradation

    print(f"\n{'='*80}")
    print(f"🔬 ICOP v7: ABLATION-VALIDATED INPUT-CONDITIONED OPTICAL PROMPT")
    print(f"{'='*80}")
    print(f"Device: {device}")
    print(f"σ={SIGMA} (moderate defocus — realistic, not image-destroying)")
    print(f"THREE models compared:")
    print(f"  1. Baseline         — clean pretrained, no adaptation")
    print(f"  2. FT-Baseline      — clean pretrained + mixed head FT (null hypothesis)")
    print(f"  3. ICOP             — clean pretrained + prompt evolution + mixed FT")
    print(f"Primary metric: ICOP blurred acc vs FT-Baseline blurred acc")
    print(f"{'='*80}\n")

    all_results = {}
    for ds in ['mnist', 'fashion', 'cifar10']:
        print(f"\n{'█'*80}\n█ DATASET: {ds.upper()}\n{'█'*80}")
        all_results[ds] = validate_icop(
            dataset_name=ds, num_val=5000, device=device, blur_dim=32, sigma=SIGMA)

    # ── Results table ──────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"📊 RESULTS: 3-WAY COMPARISON (σ={SIGMA})")
    print(f"{'='*80}")
    print(f"{'Dataset':<12} {'Model':<16} {'Blurred Acc':>12} {'Clean Acc':>10} {'Deg':>8}")
    print(f"{'-'*65}")

    for ds, res in all_results.items():
        b  = res['baseline']
        ft = res['ft_baseline']
        o  = res['icop']
        print(f"{ds.upper():<12} {'Baseline':<16} {b['blurred']:>11.1%} {b['clean']:>9.1%} {b['deg']:>7.1f}%")
        print(f"{'':<12} {'FT-Baseline':<16} {ft['blurred']:>11.1%} {ft['clean']:>9.1%} {ft['deg']:>7.1f}%"
              f"  Δvs base: {ft['blurred']-b['blurred']:>+.1%}")
        print(f"{'':<12} {'ICOP':<16} {o['blurred']:>11.1%} {o['clean']:>9.1%} {o['deg']:>7.1f}%"
              f"  Δvs FT: {o['blurred']-ft['blurred']:>+.1%}")
        p_vs_ft   = res['icop_vs_ft']['p']
        p_vs_base = res['icop_vs_base']['p']
        p_ft_base = res['ft_vs_base']['p']
        print(f"  McNemar: ICOP vs FT-Base p={p_vs_ft:.4f} {'✅' if p_vs_ft<0.05 else '⚠️ '} | "
              f"ICOP vs Base p={p_vs_base:.4f} {'✅' if p_vs_base<0.05 else '⚠️ '} | "
              f"FT vs Base p={p_ft_base:.4f} {'✅' if p_ft_base<0.05 else '⚠️ '}")
        print(f"{'-'*65}")

    # Summary — primary criterion: ICOP beats FT-Baseline
    icop_beats_ft  = all(r['icop_vs_ft']['delta'] > 0 for r in all_results.values())
    icop_sig_vs_ft = all(r['icop_vs_ft']['p'] < 0.05  for r in all_results.values())
    avg_delta_ft   = np.mean([r['icop_vs_ft']['delta'] for r in all_results.values()])

    print(f"\n📈 ICOP vs FT-Baseline (prompt contribution, primary):")
    print(f"   Average Δ blurred acc: {avg_delta_ft:+.1%}")
    print(f"   All positive:          {'✅ Yes' if icop_beats_ft  else '⚠️  No'}")
    print(f"   All p<0.05:            {'✅ Yes' if icop_sig_vs_ft else '⚠️  No'}")

    torch.save(all_results, 'icop_v7_results.pth')
    print(f"\n💾 Saved to: icop_v7_results.pth")


🔬 ICOP v7: ABLATION-VALIDATED INPUT-CONDITIONED OPTICAL PROMPT
Device: cuda
σ=1.5 (moderate defocus — realistic, not image-destroying)
THREE models compared:
  1. Baseline         — clean pretrained, no adaptation
  2. FT-Baseline      — clean pretrained + mixed head FT (null hypothesis)
  3. ICOP             — clean pretrained + prompt evolution + mixed FT
Primary metric: ICOP blurred acc vs FT-Baseline blurred acc


████████████████████████████████████████████████████████████████████████████████
█ DATASET: MNIST
████████████████████████████████████████████████████████████████████████████████

🚀 VALIDATING ICOP v7 ON MNIST (σ=1.5)



100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 480kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.67MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.19MB/s]


Train: 60000 | Test: 10000 | Val: 5000 per condition | σ=1.5

[1/5] BASELINE: clean pretraining (60 epochs)...


  Epoch 15: Loss=0.0241, Acc=99.3%


  Epoch 30: Loss=0.0112, Acc=99.7%


...

  Epoch 60: Loss=0.0001, Acc=100.0%
  ✓ ICOP pretraining complete

[4/5] Evolving input-conditioned prompt...
  Evolving ICOP (σ=1.5, margin=0.6, iters=400)...
    Iter  100/400 | Loss:6.8660 | corr=0.1118 sup=0.0515 margin=0.0000 | ‖p_blur‖=5.633 ‖p_cln‖=3.595 (gap=+2.038) | Err↓:35.5%
    Iter  200/400 | Loss:6.2721 | corr=0.1114 sup=0.0161 margin=0.0000 | ‖p_blur‖=5.789 ‖p_cln‖=1.861 (gap=+3.929) | Err↓:38.5%
    Iter  300/400 | Loss:6.6987 | corr=0.1185 sup=0.0128 margin=0.0000 | ‖p_blur‖=6.280 ‖p_cln‖=1.628 (gap=+4.652) | Err↓:36.8%
    Iter  400/400 | Loss:6.1053 | corr=0.1094 sup=0.0121 margin=0.0000 | ‖p_blur‖=5.711 ‖p_cln‖=1.561 (gap=+4.150) | Err↓:41.8%

  ✓ ICOP evolved (best loss: 4.0881)

[5/5] ICOP fine-tuning: backbone FROZEN, mixed data (30 epochs)...


  [ICOP-FT] Epoch 10: ValBlur=56.2% (best=56.2%)


  [ICOP-FT] Epoch 20: ValBlur=57.5% (best=57.5%)


  [ICOP-FT] Epoch 30: ValBlur=58.0% (best=58.0%)



  ✓ ICOP: Clean=83.4%, Blurred=58.0%, Degradation=30.5%


📊 RESULTS: 3-WAY COMPARISON (σ=1.5)
Dataset      Model             Blurred Acc  Clean Acc      Deg
-----------------------------------------------------------------
MNIST        Baseline               20.6%     99.4%    79.2%
             FT-Baseline            95.3%     98.9%     3.6%  Δvs base: +74.7%
             ICOP                   98.0%     99.3%     1.3%  Δvs FT: +2.7%
  McNemar: ICOP vs FT-Base p=0.0000 ✅ | ICOP vs Base p=0.0000 ✅ | FT vs Base p=0.0000 ✅
-----------------------------------------------------------------
FASHION      Baseline               43.8%     91.4%    52.1%
             FT-Baseline            75.9%     90.3%    16.0%  Δvs base: +32.1%
             ICOP                   84.4%     90.7%     6.9%  Δvs FT: +8.6%
  McNemar: ICOP vs FT-Base p=0.0000 ✅ | ICOP vs Base p=0.0000 ✅ | FT vs Base p=0.0000 ✅
-----------------------------------------------------------------
CIFAR10      Baseline               